# WAFER-GUARD · Fab-7 Excursion Detector — repaired pipeline

In [1]:
import os, json, copy, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             roc_auc_score, average_precision_score, confusion_matrix)
import joblib

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
SEED = 1337
set_seed(SEED)
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True, warn_only=True)
DEVICE = torch.device("cpu")

HP = dict(test_frac=0.20, val_frac=0.20, batch_size=64, max_epochs=60, patience=10,
          lr=1e-3, weight_decay=1e-4, hidden=256, latent=32, dropout=0.10, clip_z=10.0)
GRID_LAMBDA = [0.5, 1.0, 5.0]          # weight of the classification loss, chosen on validation only
GRID_SEEDS = [1337, 1338, 1339]
LEGACY_CODES = [100.0, 200.0, 300.0]   # firmware error codes of legacy station PS-L
GATE_F1 = 0.90
print(f"torch {torch.__version__} | device={DEVICE}")

torch 2.14.1+cu130 | device=cpu


## 1 · Data and chronological split (train → validation → test)

In [2]:
df = pd.read_csv("secom_telemetry_modified.csv", parse_dates=["timestamp"])
SENSOR_COLS = [c for c in df.columns if c.startswith("sensor_")]
assert len(SENSOR_COLS) == 590 and SENSOR_COLS[:2] == ["sensor_0", "sensor_1"]
assert set(df["label"].unique()) == {-1, 1}          # -1 = pass, +1 = fail

df = df.sort_values("timestamp", kind="mergesort").reset_index(drop=True)
N = len(df)
test_start = int((1 - HP["test_frac"]) * N)
val_start = int((1 - HP["val_frac"]) * test_start)
parts = {"train": df.iloc[:val_start], "val": df.iloc[val_start:test_start], "test": df.iloc[test_start:]}
assert parts["train"].timestamp.max() <= parts["val"].timestamp.min()
assert parts["val"].timestamp.max() <= parts["test"].timestamp.min()

split_table = pd.DataFrame([dict(part=k, n=len(p), defects=int((p.label == 1).sum()),
                                 defect_rate=f"{(p.label == 1).mean():.2%}",
                                 start=p.timestamp.min(), end=p.timestamp.max()) for k, p in parts.items()])
print(split_table.to_string(index=False))

 part    n  defects defect_rate               start                 end
train 1002       76       7.58% 2008-07-19 11:55:00 2008-09-22 00:28:00
  val  251       11       4.38% 2008-09-22 00:47:00 2008-10-02 19:25:00
 test  314       17       5.41% 2008-10-02 20:54:00 2008-10-17 06:07:00


## 2 · Legacy probe station PS-L: flag + categorical codes

In [3]:
def raw_features(part):
    X = part[SENSOR_COLS].to_numpy(dtype=np.float64).copy()
    s0, s1 = X[:, 0].copy(), X[:, 1].copy()
    legacy = (s0 == 0.0)                                   # sentinel
    assert np.array_equal(legacy, np.isin(s1, LEGACY_CODES)), "sentinel and error-code signatures disagree"
    X[legacy, 0] = np.nan                                  # not a measurement
    X[legacy, 1] = np.nan                                  # error code, not a measurement
    onehot = np.stack([legacy & (s1 == c) for c in LEGACY_CODES], axis=1).astype(np.float32)
    y = (part["label"].to_numpy() == 1).astype(np.int64)
    return X, legacy.astype(np.float32), onehot, y

raw = {k: raw_features(p) for k, p in parts.items()}
for k, (X, leg, oh, y) in raw.items():
    print(f"{k:5s}: legacy={int(leg.sum()):3d} | defects among legacy={int(y[leg == 1].sum())} | codes 100/200/300={oh.sum(0).astype(int).tolist()}")

train: legacy= 85 | defects among legacy=0 | codes 100/200/300=[34, 24, 27]
val  : legacy= 34 | defects among legacy=0 | codes 100/200/300=[15, 6, 13]
test : legacy= 27 | defects among legacy=0 | codes 100/200/300=[12, 5, 10]


## 3 · Preprocessing fitted on training rows only

In [4]:
class Preprocessor:
    def __init__(self, clip): self.clip = clip
    def fit(self, X_train):
        sd = np.nanstd(X_train, axis=0)
        self.keep = np.where(np.isfinite(sd) & (sd > 0))[0]          # drop columns constant in training
        self.scaler = StandardScaler().fit(X_train[:, self.keep])    # NaNs ignored
        return self
    def transform(self, X):
        Z = self.scaler.transform(X[:, self.keep])
        M = (~np.isnan(Z)).astype(np.float32)                        # 1 = really measured
        Z = np.clip(np.nan_to_num(Z, nan=0.0), -self.clip, self.clip)  # missing / not-applicable -> training mean (0 after scaling)
        return Z.astype(np.float32), M

pre = Preprocessor(HP["clip_z"]).fit(raw["train"][0])
F_CONT = len(pre.keep)
POS_S0 = int(np.where(pre.keep == 0)[0][0]); POS_S1 = int(np.where(pre.keep == 1)[0][0])
print(f"continuous features kept: {F_CONT} of 590")

def to_tensors(key):
    X, leg, oh, y = raw[key]
    Z, M = pre.transform(X)
    return dict(x=torch.from_numpy(np.hstack([Z, leg[:, None], oh]).astype(np.float32)),
                target=torch.from_numpy(Z), mask=torch.from_numpy(M), y=torch.from_numpy(y.astype(np.float32)))

T = {k: to_tensors(k) for k in raw}
D_IN, D_OUT = T["train"]["x"].shape[1], T["train"]["target"].shape[1]
for k, t in T.items():
    print(f"{k:5s}: x{tuple(t['x'].shape)} target{tuple(t['target'].shape)} mask{tuple(t['mask'].shape)} y{tuple(t['y'].shape)}")

continuous features kept: 468 of 590


train: x(1002, 472) target(1002, 468) mask(1002, 468) y(1002,)
val  : x(251, 472) target(251, 468) mask(251, 468) y(251,)
test : x(314, 472) target(314, 468) mask(314, 468) y(314,)


In [5]:
# causality checks
col = int(pre.keep[10])
assert np.isclose(pre.scaler.mean_[10], np.nanmean(raw["train"][0][:, col]))
assert not np.isclose(pre.scaler.mean_[10], np.nanmean(np.vstack([raw[k][0] for k in raw])[:, col]))
assert np.nanmax(pre.scaler.n_samples_seen_) <= len(parts["train"])
Z_all, M_all = pre.transform(raw["test"][0])
for i in [0, 7, 100, len(Z_all) - 1]:                     # a test row's features must not depend on other rows
    Z_i, M_i = pre.transform(raw["test"][0][i:i + 1])
    assert np.array_equal(Z_all[i], Z_i[0]) and np.array_equal(M_all[i], M_i[0])
# legacy cells are marked 'not measured', never scaled as continuous values
leg_rows = np.where(T["train"]["x"][:, F_CONT].numpy() == 1)[0]
assert (T["train"]["mask"][leg_rows][:, [POS_S0, POS_S1]] == 0).all()
print("causality / legacy checks passed")

causality / legacy checks passed


## 4 · Model and loss

In [6]:
class WaferSAE(nn.Module):
    def __init__(self, d_in, d_out, d_hid, d_lat, p):
        super().__init__()
        self.d_in, self.d_out = d_in, d_out
        self.encoder = nn.Sequential(
            nn.Linear(d_in, d_hid), nn.LayerNorm(d_hid), nn.GELU(), nn.Dropout(p),
            nn.Linear(d_hid, d_hid // 2), nn.LayerNorm(d_hid // 2), nn.GELU(),
            nn.Linear(d_hid // 2, d_lat))
        self.decoder = nn.Sequential(
            nn.Linear(d_lat, d_hid // 2), nn.GELU(),
            nn.Linear(d_hid // 2, d_hid), nn.GELU(),
            nn.Linear(d_hid, d_out))
        self.head = nn.Linear(d_lat, 1)

    def forward(self, x):
        assert x.ndim == 2 and x.shape[1] == self.d_in, f"expected [B,{self.d_in}], got {tuple(x.shape)}"
        z = self.encoder(x)
        recon, logit = self.decoder(z), self.head(z).squeeze(-1)
        assert recon.shape == (x.shape[0], self.d_out) and logit.shape == (x.shape[0],)
        return recon, logit, z

def masked_mse(recon, target, mask):
    # sample i is compared only with sample i, and only over genuinely measured cells
    assert recon.shape == target.shape == mask.shape and recon.ndim == 2, \
        f"shape mismatch: {tuple(recon.shape)} {tuple(target.shape)} {tuple(mask.shape)}"
    return ((recon - target) ** 2 * mask).sum() / mask.sum().clamp(min=1.0)

In [7]:
# loss unit tests
torch.manual_seed(0)
B, Fd = 8, 20
t = torch.randn(B, Fd); m = (torch.rand(B, Fd) > 0.2).float(); r = torch.randn(B, Fd, requires_grad=True)
assert masked_mse(t.clone().requires_grad_(), t, m).item() == 0.0
loss = masked_mse(r, t, m)
slow = sum(((r[i, j] - t[i, j]) ** 2 * m[i, j]).item() for i in range(B) for j in range(Fd)) / m.sum().item()
assert abs(loss.item() - slow) < 1e-5
loss.backward()
assert torch.allclose(r.grad, 2 * (r.detach() - t) * m / m.sum(), atol=1e-6) and (r.grad[m == 0] == 0).all()
try:
    masked_mse(r.detach(), t.unsqueeze(1), m); raise RuntimeError("old [B,1,F] call was not rejected")
except AssertionError:
    pass
print("loss unit tests passed")

loss unit tests passed


## 5 · Training with a class-weighted loss (no resampling)

In [8]:
n_pos = float(T["train"]["y"].sum()); n_neg = float(len(T["train"]["y"]) - n_pos)
POS_WEIGHT = torch.tensor(n_neg / n_pos)
print(f"train: {int(n_pos)} defects / {int(n_neg)} pass -> pos_weight = {POS_WEIGHT.item():.2f}")

def make_loader(seed):
    g = torch.Generator().manual_seed(seed); t = T["train"]
    return DataLoader(TensorDataset(t["x"], t["target"], t["mask"], t["y"]),
                      batch_size=HP["batch_size"], shuffle=True, generator=g)

@torch.no_grad()
def predict(model, part):
    model.eval()
    recon, logit, _ = model(T[part]["x"].to(DEVICE))
    return torch.sigmoid(logit).cpu().numpy(), masked_mse(recon.cpu(), T[part]["target"], T[part]["mask"]).item()

def train_one(lambda_clf, seed):
    set_seed(seed)
    model = WaferSAE(D_IN, D_OUT, HP["hidden"], HP["latent"], HP["dropout"]).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=HP["lr"], weight_decay=HP["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=HP["max_epochs"])
    bce = nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT.to(DEVICE))
    loader = make_loader(seed)
    best = dict(val_ap=-1.0, epoch=0, state=None, val_recon=None); wait, epochs_run = 0, 0
    for epoch in range(1, HP["max_epochs"] + 1):
        model.train()
        for xb, tb, mb, yb in loader:
            xb, tb, mb, yb = xb.to(DEVICE), tb.to(DEVICE), mb.to(DEVICE), yb.to(DEVICE)
            recon, logit, _ = model(xb)
            assert logit.shape == yb.shape
            loss = masked_mse(recon, tb, mb) + lambda_clf * bce(logit, yb)
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        sched.step(); epochs_run = epoch
        pv, vrec = predict(model, "val")
        vap = average_precision_score(T["val"]["y"].numpy(), pv)        # selection metric: validation AP
        if vap > best["val_ap"]:
            best.update(val_ap=vap, epoch=epoch, state=copy.deepcopy(model.state_dict()), val_recon=vrec); wait = 0
        else:
            wait += 1
            if wait >= HP["patience"]: break
    best["epochs_run"] = epochs_run
    return best

# one forward/backward pass: shapes are right and gradients reach every part of the model
_m = WaferSAE(D_IN, D_OUT, HP["hidden"], HP["latent"], HP["dropout"])
_xb, _tb, _mb, _yb = [t[:16] for t in (T["train"]["x"], T["train"]["target"], T["train"]["mask"], T["train"]["y"])]
_r, _l, _ = _m(_xb)
(masked_mse(_r, _tb, _mb) + nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT)(_l, _yb)).backward()
for name, mod in [("encoder", _m.encoder), ("decoder", _m.decoder), ("head", _m.head)]:
    assert all(p.grad is not None and p.grad.abs().sum() > 0 for p in mod.parameters()), f"no gradient reaches {name}"
print("shapes:", tuple(_xb.shape), "->", tuple(_r.shape), tuple(_l.shape), "| gradients reach encoder, decoder and head")

train: 76 defects / 926 pass -> pos_weight = 12.18


shapes: (16, 472) -> (16, 468) (16,) | gradients reach encoder, decoder and head


In [9]:
runs = []
for lam in GRID_LAMBDA:
    for sd in GRID_SEEDS:
        b = train_one(lam, sd)
        runs.append(dict(lambda_clf=lam, seed=sd, best_epoch=b["epoch"], val_ap=b["val_ap"], state=b["state"]))
        print(f"lambda_clf={lam:<4} seed={sd} | best epoch {b['epoch']:2d} (ran {b['epochs_run']:2d}) | val AP={b['val_ap']:.4f}")

grid = pd.DataFrame([{k: v for k, v in r.items() if k != "state"} for r in runs])
mean_ap = grid.groupby("lambda_clf")["val_ap"].mean()
best_lambda = float(mean_ap.idxmax())
final = max([r for r in runs if r["lambda_clf"] == best_lambda], key=lambda r: r["val_ap"])
print(f"selected on validation only: lambda_clf={final['lambda_clf']} seed={final['seed']} epoch={final['best_epoch']} val AP={final['val_ap']:.4f}")
final_model = WaferSAE(D_IN, D_OUT, HP["hidden"], HP["latent"], HP["dropout"]).to(DEVICE)
final_model.load_state_dict(final["state"])

lambda_clf=0.5  seed=1337 | best epoch  1 (ran 11) | val AP=0.0514


lambda_clf=0.5  seed=1338 | best epoch  3 (ran 13) | val AP=0.0540


lambda_clf=0.5  seed=1339 | best epoch  1 (ran 11) | val AP=0.0739


lambda_clf=1.0  seed=1337 | best epoch  1 (ran 11) | val AP=0.0514


lambda_clf=1.0  seed=1338 | best epoch  7 (ran 17) | val AP=0.0609


lambda_clf=1.0  seed=1339 | best epoch  1 (ran 11) | val AP=0.0727


lambda_clf=5.0  seed=1337 | best epoch  7 (ran 17) | val AP=0.0523


lambda_clf=5.0  seed=1338 | best epoch  6 (ran 16) | val AP=0.0676


lambda_clf=5.0  seed=1339 | best epoch  1 (ran 11) | val AP=0.0718
selected on validation only: lambda_clf=5.0 seed=1339 epoch=1 val AP=0.0718


<All keys matched successfully>

## 6 · Decision threshold (validation only, then frozen)

In [10]:
yv = T["val"]["y"].numpy().astype(int)
p_val, _ = predict(final_model, "val")

def best_threshold(y_true, prob):
    best_t, best_f = 0.5, -1.0
    for t in np.unique(prob):
        f = f1_score(y_true, (prob >= t).astype(int), zero_division=0)
        if f > best_f or (f == best_f and t > best_t): best_t, best_f = float(t), float(f)
    return best_t, best_f

THRESHOLD, val_f1 = best_threshold(yv, p_val)
FROZEN = dict(threshold=THRESHOLD, lambda_clf=final["lambda_clf"], seed=final["seed"], epoch=final["best_epoch"])
print(f"validation: {int(yv.sum())} defects of {len(yv)} | AP={average_precision_score(yv, p_val):.4f} "
      f"AUC={roc_auc_score(yv, p_val):.4f} | threshold={THRESHOLD:.4f} (val F1={val_f1:.4f})")

validation: 11 defects of 251 | AP=0.0718 AUC=0.6004 | threshold=0.5832 (val F1=0.1429)


## 7 · Final evaluation on the untouched last 20 % (natural prevalence, run once)

In [11]:
yt = T["test"]["y"].numpy().astype(int)
p_test, _ = predict(final_model, "test")
pred = (p_test >= FROZEN["threshold"]).astype(int)
tn, fp, fn, tp = confusion_matrix(yt, pred, labels=[0, 1]).ravel()
metrics = dict(n_test=int(len(yt)), n_defects_test=int(yt.sum()), test_prevalence=float(yt.mean()),
               threshold=FROZEN["threshold"], flagged=int(pred.sum()),
               precision=float(precision_score(yt, pred, zero_division=0)),
               recall=float(recall_score(yt, pred, zero_division=0)),
               f1=float(f1_score(yt, pred, zero_division=0)),
               roc_auc=float(roc_auc_score(yt, p_test)),
               average_precision=float(average_precision_score(yt, p_test)),
               tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp),
               no_skill_ap=float(yt.mean()), f1_if_flag_everything=float(2 * yt.mean() / (1 + yt.mean())))

# bootstrap 95 % intervals (the test set has very few defects)
rng = np.random.default_rng(SEED); boot = {"roc_auc": [], "average_precision": []}
for _ in range(2000):
    ii = rng.integers(0, len(yt), len(yt))
    if yt[ii].sum() == 0: continue
    boot["roc_auc"].append(roc_auc_score(yt[ii], p_test[ii]))
    boot["average_precision"].append(average_precision_score(yt[ii], p_test[ii]))
metrics["bootstrap_95ci"] = {k: [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))] for k, v in boot.items()}

print(f"TEST: n={metrics['n_test']} | defective={metrics['n_defects_test']} | prevalence={metrics['test_prevalence']:.2%} | flagged={metrics['flagged']}")
print(f"precision {metrics['precision']:.4f} | recall {metrics['recall']:.4f} | F1 {metrics['f1']:.4f} | "
      f"ROC-AUC {metrics['roc_auc']:.4f} | AP {metrics['average_precision']:.4f}")
print(f"confusion matrix  TN={tn}  FP={fp}  FN={fn}  TP={tp}")
print(f"no-skill AP = {metrics['no_skill_ap']:.4f} | F1 if every wafer were flagged = {metrics['f1_if_flag_everything']:.4f}")
for k, (lo, hi) in metrics["bootstrap_95ci"].items():
    print(f"bootstrap 95% CI {k}: [{lo:.3f}, {hi:.3f}]")
passed = metrics["f1"] >= GATE_F1
print(f"Release gate (F1 >= {GATE_F1}): {'MET' if passed else 'NOT MET'}")

TEST: n=314 | defective=17 | prevalence=5.41% | flagged=5
precision 0.0000 | recall 0.0000 | F1 0.0000 | ROC-AUC 0.5722 | AP 0.1175
confusion matrix  TN=292  FP=5  FN=17  TP=0
no-skill AP = 0.0541 | F1 if every wafer were flagged = 0.1027
bootstrap 95% CI roc_auc: [0.405, 0.740]
bootstrap 95% CI average_precision: [0.051, 0.270]
Release gate (F1 >= 0.9): NOT MET


In [12]:
os.makedirs("artifacts", exist_ok=True)
torch.save(final_model.state_dict(), "artifacts/wafer_sae_fixed.pt")
joblib.dump(dict(preprocessor=pre, legacy_codes=LEGACY_CODES, hp=HP, frozen=FROZEN,
                 feature_columns=[SENSOR_COLS[i] for i in pre.keep],
                 input_layout="[scaled sensors..., is_legacy, code_100, code_200, code_300]"),
            "artifacts/preprocessor_fixed.joblib")
results = dict(split=json.loads(split_table.astype({"start": str, "end": str}).to_json(orient="records")),
               pos_weight=float(POS_WEIGHT), n_features_kept=int(F_CONT), selected=FROZEN,
               grid=json.loads(grid.to_json(orient="records")),
               val=dict(prevalence=float(yv.mean()), n_defects=int(yv.sum()), f1_at_threshold=val_f1,
                        ap=float(average_precision_score(yv, p_val)), auc=float(roc_auc_score(yv, p_val))),
               test=metrics, gate_f1=GATE_F1, gate_met=bool(passed))
with open("artifacts/results.json", "w") as f: json.dump(results, f, indent=2)
print("saved artifacts/")

saved artifacts/
